<a href="https://colab.research.google.com/github/22009161/Executive-Business-Intelligence-Dashboarding-Power-BI-Advanced-Excel-/blob/main/Another_copy_of_Relational_Database_Design_%26_SQL_Performance_Analytics_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import pandas as pd

# =========================
# CREATE DATABASE
# =========================
conn = sqlite3.connect("enterprise_bi.db")
cursor = conn.cursor()

cursor.executescript("""

DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS employees;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    full_name TEXT,
    region TEXT,
    segment TEXT
);

CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    employee_name TEXT,
    department TEXT
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    unit_price REAL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    employee_id INTEGER,
    order_date DATE,
    total_amount REAL
);

CREATE TABLE order_items (
    item_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    subtotal REAL
);

-- =========================
-- DATA INSERTION
-- =========================
INSERT INTO customers VALUES
(1,'John Doe','Gauteng','Corporate'),
(2,'Jane Smith','Western Cape','SME'),
(3,'Sipho Dlamini','KZN','Retail'),
(4,'Ayesha Khan','Gauteng','Corporate');

INSERT INTO employees VALUES
(1,'David Mokoena','Sales'),
(2,'Sarah Williams','Sales'),
(3,'Thabo Nkosi','Sales');

INSERT INTO products VALUES
(1,'Laptop','Electronics',15000),
(2,'Phone','Electronics',8000),
(3,'Headphones','Accessories',1200),
(4,'Keyboard','Accessories',900);

INSERT INTO orders VALUES
(1,1,1,'2025-04-01',15000),
(2,2,2,'2025-04-03',9200),
(3,3,3,'2025-04-05',1200),
(4,4,1,'2025-04-07',10100);

INSERT INTO order_items VALUES
(1,1,1,1,15000),
(2,2,2,1,8000),
(3,2,3,1,1200),
(4,3,3,1,1200),
(5,4,2,1,8000),
(6,4,4,2,1800);

""")

conn.commit()

print("DATABASE CREATED")

# =========================
# KPI ANALYSIS
# =========================
kpi = pd.read_sql_query("""
SELECT
    SUM(total_amount) AS total_revenue,
    COUNT(order_id) AS total_orders,
    AVG(total_amount) AS avg_order_value
FROM orders
""", conn)

print("\nKPI METRICS")
print(kpi)

# =========================
# EMPLOYEE PERFORMANCE
# =========================
emp = pd.read_sql_query("""
SELECT
    e.employee_name,
    SUM(o.total_amount) AS revenue
FROM employees e
JOIN orders o ON e.employee_id = o.employee_id
GROUP BY e.employee_id
ORDER BY revenue DESC
""", conn)

print("\nEMPLOYEE PERFORMANCE")
print(emp)

# =========================
# PRODUCT PERFORMANCE
# =========================
prod = pd.read_sql_query("""
SELECT
    p.product_name,
    SUM(oi.quantity) AS units_sold,
    SUM(oi.subtotal) AS revenue
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
GROUP BY p.product_id
ORDER BY revenue DESC
""", conn)

print("\nPRODUCT PERFORMANCE")
print(prod)

# =========================
# SEGMENT ANALYSIS
# =========================
seg = pd.read_sql_query("""
SELECT
    segment,
    SUM(o.total_amount) AS revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY segment
""", conn)

print("\nSEGMENT ANALYSIS")
print(seg)

# =========================
# EXPORT FOR POWER BI
# =========================
customers = pd.read_sql_query("SELECT * FROM customers", conn)
employees = pd.read_sql_query("SELECT * FROM employees", conn)
products = pd.read_sql_query("SELECT * FROM products", conn)
orders = pd.read_sql_query("SELECT * FROM orders", conn)
order_items = pd.read_sql_query("SELECT * FROM order_items", conn)

customers.to_csv("customers.csv", index=False)
employees.to_csv("employees.csv", index=False)
products.to_csv("products.csv", index=False)
orders.to_csv("orders.csv", index=False)
order_items.to_csv("order_items.csv", index=False)

print("\nPOWER BI FILES EXPORTED")

conn.close()

DATABASE CREATED

KPI METRICS
   total_revenue  total_orders  avg_order_value
0        35500.0             4           8875.0

EMPLOYEE PERFORMANCE
    employee_name  revenue
0   David Mokoena  25100.0
1  Sarah Williams   9200.0
2     Thabo Nkosi   1200.0

PRODUCT PERFORMANCE
  product_name  units_sold  revenue
0        Phone           2  16000.0
1       Laptop           1  15000.0
2   Headphones           2   2400.0
3     Keyboard           2   1800.0

SEGMENT ANALYSIS
     segment  revenue
0  Corporate  25100.0
1     Retail   1200.0
2        SME   9200.0

POWER BI FILES EXPORTED
